In [ ]:
'''
Simulate the auditory neurons using the model.sim_config.py 
New scripts I added:
simulate.Auditory_neurons.py 
    Class to simulate O-shaped (Difference of Gaussian) neural tuning curves in response to auditory stimuli (e.g. frequency and level
    as measured for Frequency Response Area)

for ground truth plots of these Auditory neuron tuning curves (three types), 
    plot_tuningcurves(N, SimPop, config) (found in plot.py, called in run_simulations)

model.grid_sample.py 
    the center of the stimulus space is finely sampled, note that x_coarse and x_fine have to be adjusted manually
    for a dimension size beyond d1 = 10, d2 = 10

simulate.pareto_plot.py

simulate.sampling_plots.py  
            samples the entire stimulus space using sample function from Auditory_neurons class, sanity check 
            plot for comparing true and sampled peak

simulate.stop_functions_plots.py   
    plots the stopping functions: takes the stopping value (Expected Improvement value from the results_dict output 
    from bayesopt_sampling for each test for each neuron, allows a comparison of different runs of the simulation 
    as a function of gamma in one plot and nu in another plot


Modified model.bayesopt_sampling.py to have a second way of counting a neuron as "correct"
    Pr_list_correct_solution: probabilities of correct predictions even if EI is still above stopping criteria 
    using Bayes Opt sampling
    Pr_list: both correct peak AND EI < stopping_crit

Modified parameters_indep.yml to include a mse_cutoff value, so that random and grid sampling scripts can
follow the same stopping requirements

IN PROGRESS: 
fixing in save_results my indexing of stopping_allN for my offline Gaussian fit data, still dealing with some
inhomogeneous outputs


'''

#Main commands I added to run_simulations
from model.random_sampling import random_sampling
from model.grid_sample import grid_sampling
from simulate.sampling_plots import plot_tuningcurves_sampled_auditory, sampling_for_plots_auditory
import matplotlib.pyplot as plt
from simulate.stop_functions_plots import stop_functions_plots
from simulate.pareto_plot import gather_hyperparameters, plot_pareto_from_data


SimPop = config.SimPop
plot_tuningcurves(N, SimPop, config) #Ground truth plot (the three panel figure featuring the Gaussians that were subtracted)
plot_tuningcurves_sampled_auditory(config) #Sampling Sanity check plot (samples the whole stimulus space)

rsPr_list = random_sampling(config, print_flag=True)
gsPr_list = grid_sampling(config, print_flag=True)

#to see the stopping function with a subplot per neuron
plot_stopping_criteria(stopping_allN =results_dict['stopping_allN'], stopping_crit = float(config.optimizers['optim_1']['stopping_crit']), EI_or_PI = "EI")

#to see stopping as a function of neuron number and gamma or nu
stop_functions_plots(section_key='stopping_allN')

#Plotting Pareto
run_data = gather_hyperparameters(output_dir='output', section_key='Pr_list')
plot_pareto_from_data(run_data)